## 1. Configuração inicial

Ativa o matplotlib interativo (`%matplotlib widget`, necessário para clicar no tabuleiro), importa as bibliotecas e define o tamanho do tabuleiro (6x6 pontos), as cores dos jogadores e do desenho (CORES_JOGO) e as cores do placar (CORES_PLACAR).

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

TAMANHO = 6

CORES_JOGO = {
    "jogador_1": "#89759C",
    "jogador_2": "#D5BC7E",
    "fundo": "#F7F5F2",
    "disponivel": "#D3CED8",
    "pontos": "#3B363F"
}

CORES_PLACAR = {
    "texto": "#3B363F",
    "texto_suave": "#6E6676",
    "texto_legenda": "#8A8292",
    "borda": "#E3DEE7",
    "fundo_cartao": "#FFFFFF",
}

## 2. Tabuleiro

Cria a matriz 11x11 que guarda o estado do jogo. As posições (par, par) são os pontos, (par, ímpar) as linhas horizontais, (ímpar, par) as linhas verticais e (ímpar, ímpar) os quadrados. O valor 0 significa livre, 1 é do Jogador 1 e 2 é do Jogador 2.

In [ ]:
def criar_tabuleiro():
    tabuleiro = np.zeros((11, 11), dtype=int)
    return tabuleiro

tabuleiro = criar_tabuleiro()

## 3. Desenho do tabuleiro

Define a função `desenhar_tabuleiro`, que desenha os pontos, as linhas (tracejadas quando livres, sólidas na cor do jogador quando marcadas) e os quadrados conquistados, pintados com a cor de quem os fechou.

In [ ]:
def desenhar_tabuleiro(tabuleiro):
    fig, ax = plt.subplots(figsize=(8, 8))
    fig.patch.set_facecolor(CORES_JOGO["fundo"])
    ax.set_facecolor(CORES_JOGO["fundo"])

    for i in range(6):
        for j in range(6):

            # Pontos
            ax.add_patch(
                Circle(
                    (j, 5 - i),
                    radius=0.065,
                    color=CORES_JOGO["pontos"],
                    zorder=3
                )
            )

            # Linhas horizontais
            if j < 5:
                valor = tabuleiro[2*i, 2*j + 1]

                cor = (
                    CORES_JOGO["jogador_1"] if valor == 1 else
                    CORES_JOGO["jogador_2"] if valor == 2 else
                    CORES_JOGO["disponivel"]
                )

                ax.plot(
                    [j, j + 1], [5-i, 5-i],
                    color=cor,
                    linewidth=5 if valor else 2,
                    linestyle="-" if valor else "--",
                    solid_capstyle="round"
                )

            # Linhas verticais
            if i < 5:
                valor = tabuleiro[2*i + 1, 2*j]

                cor = (
                    CORES_JOGO["jogador_1"] if valor == 1 else
                    CORES_JOGO["jogador_2"] if valor == 2 else
                    CORES_JOGO["disponivel"]
                )

                ax.plot(
                    [j, j], [5-i, 4-i],
                    color=cor,
                    linewidth=5 if valor else 2,
                    linestyle="-" if valor else "--",
                    solid_capstyle="round"
                )

            # Quadrados conquistados
            if i < 5 and j < 5:
                dono = tabuleiro[2*i + 1, 2*j + 1]

                if dono:
                    cor = CORES_JOGO[f"jogador_{dono}"]

                    ax.add_patch(
                        Rectangle(
                            (j + 0.08, 4-i + 0.08),
                            0.84, 0.84,
                            facecolor=cor,
                            alpha=0.30,
                            edgecolor="none",
                            zorder=0
                        )
                    )

    ax.set_xlim(-0.3, 5.3)
    ax.set_ylim(-0.3, 5.3)
    ax.set_aspect("equal")
    ax.axis("off")

    plt.title("Pontas e Quadrados", fontsize=20, color=CORES_JOGO["pontos"])
    plt.show()

## 4. Primeiro desenho

Mostra o tabuleiro vazio, só para conferir como o desenho fica antes de começar a jogar.

In [ ]:
desenhar_tabuleiro(tabuleiro)

## 5. Regras do jogo (motor)

Guarda o estado da partida (tabuleiro, jogador da vez e pontuação) e as regras: lista as jogadas possíveis, marca uma linha para o jogador da vez, fecha os quadrados que ficarem completos com a cor desse jogador e decide a vez (quem fecha um quadrado joga de novo, senão passa para o outro). Também identifica o fim do jogo, quando todas as linhas foram marcadas.

In [ ]:
# Motor

from dataclasses import dataclass
from typing import List, Tuple


@dataclass
class Estado:
    tabuleiro: np.ndarray
    jogador_atual: int
    pontuacao: List[int]

def novo_estado() -> Estado:
    return Estado(criar_tabuleiro(), 1, [0, 0])

def jogadas_legais(estado: Estado) -> List[Tuple[int, int]]:
    return [
        (r, c)
        for r in range(11)
        for c in range(11)
        if (r + c) % 2 == 1 and estado.tabuleiro[r, c] == 0
    ]

def aplicar_jogada(estado: Estado, jogada: Tuple[int, int]) -> int:
    r, c = jogada

    if not (0 <= r <= 10 and 0 <= c <= 10):
        raise ValueError("Jogada fora do tabuleiro: %s" % (jogada,))
    if (r + c) % 2 != 1:
        raise ValueError("Posicao nao e uma aresta: %s" % (jogada,))
    if estado.tabuleiro[r, c] != 0:
        raise ValueError("Aresta ja marcada: %s" % (jogada,))

    t = estado.tabuleiro
    jogador = estado.jogador_atual
    t[r, c] = jogador

    if r % 2 == 0:
        vizinhos = [(r - 1, c), (r + 1, c)]
    else:
        vizinhos = [(r, c - 1), (r, c + 1)]

    fechados = 0
    for i, j in vizinhos:
        if not (0 <= i <= 10 and 0 <= j <= 10):
            continue
        if (
            t[i, j] == 0
            and t[i - 1, j] != 0
            and t[i + 1, j] != 0
            and t[i, j - 1] != 0
            and t[i, j + 1] != 0
        ):
            t[i, j] = jogador
            fechados += 1

    estado.pontuacao[jogador - 1] += fechados

    if fechados == 0:
        estado.jogador_atual = 2 if jogador == 1 else 1
    return fechados


def eh_terminal(estado: Estado) -> bool:
    return len(jogadas_legais(estado)) == 0

## 6. Desenho do tabuleiro durante o jogo

Define a função `desenhar_em`, usada pela interface para redesenhar o tabuleiro a cada jogada. Além das linhas e dos quadrados conquistados, ela destaca o ponto selecionado e os vizinhos com os quais ainda é possível formar uma linha.

In [ ]:
# Desenho do tabuleiro durante o jogo

from typing import Optional
import ipywidgets as widgets

RAIO_CLIQUE = 0.25

def desenhar_em(ax, tabuleiro, selecionado=None, vizinhos=(), cor_selecao=None):
    ax.clear()

    for i in range(6):
        for j in range(6):

            # Pontos
            ax.add_patch(
                Circle(
                    (j, 5 - i),
                    radius=0.065,
                    color=CORES_JOGO["pontos"],
                    zorder=3
                )
            )

            # Linhas horizontais
            if j < 5:
                valor = tabuleiro[2*i, 2*j + 1]

                cor = (
                    CORES_JOGO["jogador_1"] if valor == 1 else
                    CORES_JOGO["jogador_2"] if valor == 2 else
                    CORES_JOGO["disponivel"]
                )

                ax.plot(
                    [j, j + 1], [5-i, 5-i],
                    color=cor,
                    linewidth=5 if valor else 2,
                    linestyle="-" if valor else "--",
                    solid_capstyle="round"
                )

            # Linhas verticais
            if i < 5:
                valor = tabuleiro[2*i + 1, 2*j]

                cor = (
                    CORES_JOGO["jogador_1"] if valor == 1 else
                    CORES_JOGO["jogador_2"] if valor == 2 else
                    CORES_JOGO["disponivel"]
                )

                ax.plot(
                    [j, j], [5-i, 4-i],
                    color=cor,
                    linewidth=5 if valor else 2,
                    linestyle="-" if valor else "--",
                    solid_capstyle="round"
                )

            # Quadrados conquistados
            if i < 5 and j < 5:
                dono = tabuleiro[2*i + 1, 2*j + 1]

                if dono:
                    cor = CORES_JOGO[f"jogador_{dono}"]

                    ax.add_patch(
                        Rectangle(
                            (j + 0.08, 4-i + 0.08),
                            0.84, 0.84,
                            facecolor=cor,
                            alpha=0.30,
                            edgecolor="none",
                            zorder=0
                        )
                    )

    # Ponto selecionado e vizinhos disponiveis
    for (i, j) in vizinhos:
        ax.add_patch(
            Circle(
                (j, 5 - i),
                radius=0.16,
                facecolor="none",
                edgecolor=CORES_JOGO["disponivel"],
                linewidth=2,
                zorder=2
            )
        )
    if selecionado is not None:
        i, j = selecionado
        ax.add_patch(
            Circle(
                (j, 5 - i),
                radius=0.16,
                color=cor_selecao or CORES_JOGO["pontos"],
                zorder=2
            )
        )

    ax.set_xlim(-0.3, 5.3)
    ax.set_ylim(-0.3, 5.3)
    ax.set_aspect("equal")
    ax.axis("off")

## 7. Interface do jogo

Monta o tabuleiro interativo (`JogoUI`). O jogador pode clicar em um ponto e depois no vizinho, ou arrastar de um ponto até o vizinho (uma linha de prévia acompanha o mouse). A linha formada aparece na cor do jogador da vez. A interface mostra de quem é a vez, o placar e tem o botão "Novo jogo".

In [ ]:
# Interface

class JogoUI:
    def __init__(self):
        self.estado = novo_estado()
        self.selecionado: Optional[Tuple[int, int]] = None
        self.arrastando: Optional[Tuple[int, int]] = None
        self._previa = None
        self.ultimo_fechou = 0

        with plt.ioff():
            self.fig, self.ax = plt.subplots(figsize=(7, 7))
        self.fig.patch.set_facecolor(CORES_JOGO["fundo"])
        self.ax.set_facecolor(CORES_JOGO["fundo"])
        self.fig.canvas.header_visible = False
        self.fig.canvas.toolbar_visible = False
        self.fig.canvas.mpl_connect("button_press_event", self._ao_apertar)
        self.fig.canvas.mpl_connect("motion_notify_event", self._ao_mover)
        self.fig.canvas.mpl_connect("button_release_event", self._ao_soltar)

        self.status = widgets.HTML(layout=widgets.Layout(width="100%"))
        self.botao_novo = widgets.Button(description="Novo jogo")
        self.botao_novo.on_click(lambda _b: self.novo_jogo())
        self.widget = widgets.VBox(
            [self.status, self.fig.canvas, self.botao_novo],
            layout=widgets.Layout(
                width="fit-content",
                max_width="100%",
                align_items="center",
            ),
        )
        self._redesenhar()

    # --- eventos do mouse (apertar, mover, soltar) ---

    def _ao_apertar(self, evento) -> None:
        if evento.button != 1 or evento.inaxes is not self.ax or evento.xdata is None:
            return
        self.apertar_xy(evento.xdata, evento.ydata)

    def _ao_mover(self, evento) -> None:
        if evento.inaxes is not self.ax or evento.xdata is None:
            return
        self.mover_xy(evento.xdata, evento.ydata)

    def _ao_soltar(self, evento) -> None:
        if evento.inaxes is not self.ax or evento.xdata is None:
            self.soltar_xy(None, None)
        else:
            self.soltar_xy(evento.xdata, evento.ydata)

    def _ponto_proximo(self, x: float, y: float) -> Optional[Tuple[int, int]]:
        j, i = round(x), round(5 - y)
        if not (0 <= i < TAMANHO and 0 <= j < TAMANHO):
            return None
        if (x - j) ** 2 + (5 - y - i) ** 2 > RAIO_CLIQUE ** 2:
            return None
        return (i, j)

    def clicar_xy(self, x: float, y: float) -> None:
        ponto = self._ponto_proximo(x, y)
        if ponto is not None:
            self.clicar_ponto(*ponto)

    def apertar_xy(self, x: float, y: float) -> None:
        # apertar num ponto funciona como clicar; se ele virou a origem da
        # linha, o arrasto a partir dele fica habilitado
        ponto = self._ponto_proximo(x, y)
        self.arrastando = None
        if ponto is None:
            return
        antes = self.selecionado
        self.clicar_ponto(*ponto)
        if self.selecionado == ponto and antes != ponto:
            self.arrastando = ponto

    def mover_xy(self, x: float, y: float) -> None:
        if self.arrastando is None:
            return
        i, j = self.arrastando
        alvo = self._ponto_proximo(x, y)
        if alvo is not None and alvo in self._vizinhos_livres(i, j):
            x, y = alvo[1], 5 - alvo[0]
        cor = self._cor(self.estado.jogador_atual)
        if self._previa is None:
            (self._previa,) = self.ax.plot(
                [j, x], [5 - i, y],
                color=cor,
                linewidth=5,
                linestyle="-",
                alpha=0.6,
                solid_capstyle="round",
                zorder=1,
            )
        else:
            self._previa.set_data([j, x], [5 - i, y])
        self.fig.canvas.draw_idle()

    def soltar_xy(self, x: Optional[float], y: Optional[float]) -> None:
        origem = self.arrastando
        self.arrastando = None
        if origem is None:
            return
        ponto = self._ponto_proximo(x, y) if x is not None else None
        if ponto == origem:
            # soltou no mesmo ponto (foi um clique): mantem a selecao
            self._redesenhar()
        elif ponto is not None and ponto in self._vizinhos_livres(*origem):
            self.clicar_ponto(*ponto)
        else:
            # soltou fora de um vizinho valido: cancela
            self.selecionado = None
            self._redesenhar()

    @staticmethod
    def _aresta(a: Tuple[int, int], b: Tuple[int, int]) -> Tuple[int, int]:
        return (a[0] + b[0], a[1] + b[1])

    @staticmethod
    def _vizinhos(i: int, j: int) -> List[Tuple[int, int]]:
        candidatos = [(i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)]
        return [(a, b) for a, b in candidatos if 0 <= a < TAMANHO and 0 <= b < TAMANHO]

    def _vizinhos_livres(self, i: int, j: int) -> List[Tuple[int, int]]:
        t = self.estado.tabuleiro
        return [
            v for v in self._vizinhos(i, j)
            if t[self._aresta((i, j), v)] == 0
        ]

    def clicar_ponto(self, i: int, j: int) -> None:
        if eh_terminal(self.estado):
            return
        ponto = (i, j)
        if self.selecionado is None:
            self.selecionado = ponto
        elif ponto == self.selecionado:
            self.selecionado = None
        elif ponto in self._vizinhos_livres(*self.selecionado):
            aresta = self._aresta(self.selecionado, ponto)
            self.ultimo_fechou = aplicar_jogada(self.estado, aresta)
            self.selecionado = None
        else:
            self.selecionado = ponto
        self._redesenhar()

    def novo_jogo(self) -> None:
        self.estado = novo_estado()
        self.selecionado = None
        self.arrastando = None
        self.ultimo_fechou = 0
        self._redesenhar()

    def _cor(self, dono: int) -> str:
        return CORES_JOGO["jogador_%d" % dono]

    def _calcular_resumo_placar(self) -> Tuple[int, int, int, str]:
        e = self.estado
        p1, p2 = e.pontuacao

        if eh_terminal(e):
            if p1 > p2:
                return p1, p2, 1, "Fim de jogo — Jogador 1 venceu!"
            if p2 > p1:
                return p1, p2, 2, "Fim de jogo — Jogador 2 venceu!"
            return p1, p2, 0, "Fim de jogo — Empate"

        destaque = e.jogador_atual

        if self.ultimo_fechou == 1:
            mensagem = f"Jogador {destaque} fechou 1 quadrado — joga de novo"
        elif self.ultimo_fechou > 1:
            mensagem = (
                f"Jogador {destaque} fechou {self.ultimo_fechou} "
                "quadrados — joga de novo"
            )
        else:
            mensagem = f"Vez do Jogador {destaque}"

        return p1, p2, destaque, mensagem

    def _montar_cartao_html(self, numero: int, pontos: int, destaque: int) -> str:
        cor = self._cor(numero)
        ativo = numero == destaque

        borda = cor if ativo else CORES_PLACAR["borda"]
        fundo_cartao = CORES_PLACAR["fundo_cartao"]
        fundo = (
            f"linear-gradient(135deg, {fundo_cartao} 0%, {cor}22 100%)"
            if ativo else fundo_cartao
        )
        sombra = (
            f"0 0 0 3px {cor}33, 0 4px 12px rgba(0,0,0,.10)"
            if ativo else "none"
        )
        opacidade = "1" if ativo else ".62"

        return f"""
            <div style="flex:1; min-width:0; background:{fundo};
                        border:2px solid {borda}; border-radius:14px;
                        padding:10px 14px; opacity:{opacidade};
                        box-shadow:{sombra};">
                <div style="display:flex; align-items:center; gap:8px;
                            font-size:12px; letter-spacing:.04em;
                            color:{CORES_PLACAR['texto_suave']};
                            text-transform:uppercase;">
                    <span style="width:10px; height:10px; border-radius:50%;
                                 background:{cor}; display:inline-block;"></span>
                    Jogador {numero}
                </div>
                <div style="font-size:30px; font-weight:700; line-height:1.1;
                            color:{cor};">{pontos}</div>
            </div>
        """

    def _montar_placar_html(
        self, p1: int, p2: int, destaque: int, mensagem: str
    ) -> str:
        return f"""
            <div style="width:100%; font-family:system-ui, sans-serif;">
                <div style="text-align:center; font-size:11px;
                            letter-spacing:.18em;
                            color:{CORES_PLACAR['texto_legenda']};
                            margin-bottom:6px;">PLACAR</div>

                <div style="display:flex; gap:12px; align-items:stretch;">
                    {self._montar_cartao_html(1, p1, destaque)}
                    {self._montar_cartao_html(2, p2, destaque)}
                </div>

                <div style="margin:8px auto 0 auto; width:fit-content;
                            max-width:100%; box-sizing:border-box;
                            padding:6px 14px; border-radius:999px;
                            background:{CORES_PLACAR['fundo_cartao']};
                            border:1px solid {CORES_PLACAR['borda']};
                            font-size:13px; color:{CORES_PLACAR['texto']};
                            font-weight:600;">{mensagem}</div>
            </div>
        """

    def _redesenhar(self) -> None:
        livres = (
            self._vizinhos_livres(*self.selecionado)
            if self.selecionado is not None else []
        )

        self._previa = None  # o ax.clear() de desenhar_em remove a linha de previa
        desenhar_em(
            self.ax,
            self.estado.tabuleiro,
            selecionado=self.selecionado,
            vizinhos=livres,
            cor_selecao=self._cor(self.estado.jogador_atual),
        )

        self.ax.set_title("Pontas e Quadrados", fontsize=20, color=CORES_JOGO["pontos"])
        p1, p2, destaque, mensagem = self._calcular_resumo_placar()
        self.status.value = self._montar_placar_html(
            p1, p2, destaque, mensagem
        )
        self.fig.canvas.draw_idle()

## 8. Iniciar o jogo

Cria uma partida e exibe o tabuleiro para os dois jogadores começarem. O Jogador 1 joga primeiro.

In [ ]:
# Iniciar jogo

from IPython.display import display

ui = JogoUI()
display(ui.widget)